# Challenge 2: Thai Word Segmentation

## แนวทาง
1. เทรน CRF บน LST20 dataset จาก HuggingFace (character-level features)
2. ใช้ CRF marginal probabilities + PyThaiNLP (newmm, longest) + TLTK เป็น ensemble
3. สร้าง 5 strategies แล้วเลือกอันที่ดีที่สุดด้วย heuristics
4. บังคับ BIE sequence constraints ให้ถูกต้อง

In [ ]:
!pip install -q kaggle python-crfsuite datasets pythainlp tltk scikit-learn

In [ ]:
import os
from google.colab import files

uploaded = files.upload()
os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
os.rename("kaggle.json", os.path.expanduser("~/.kaggle/kaggle.json"))
os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)

!kaggle competitions download -c super-ai-engineer-ss-6-word-segmentation
!unzip -qo super-ai-engineer-ss-6-word-segmentation.zip -d data

## Section 1: Constants & Character Features

Define character type classification for Thai text and CRF feature extraction.

In [ ]:
import csv
import time
import json
import warnings
from collections import Counter
from pathlib import Path
from typing import Dict, List, Set, Sequence, Tuple

warnings.filterwarnings("ignore")

import pycrfsuite
import pandas as pd

DATA_DIR = Path("data")
TEST_FILE = DATA_DIR / "ws_test.txt"
CRF_MODEL = Path("wordseg_crf.model")

LABELS = ["B_WORD", "I_WORD", "E_WORD"]

def log(msg):
    print(msg, flush=True)

# ฟังก์ชันจำแนกประเภทตัวอักษรไทย
def char_type(c):
    cp = ord(c)
    if 0x0E01 <= cp <= 0x0E2E: return "CONSONANT"
    if cp in (0x0E30, 0x0E32, 0x0E33): return "VOWEL_FOLLOWING"
    if cp in (0x0E40, 0x0E41, 0x0E42, 0x0E43, 0x0E44): return "VOWEL_LEADING"
    if 0x0E34 <= cp <= 0x0E3A: return "VOWEL_ABOVE_BELOW"
    if 0x0E47 <= cp <= 0x0E4E: return "DIACRITIC"
    if 0x0E48 <= cp <= 0x0E4B: return "TONE_MARK"
    if 0x0E50 <= cp <= 0x0E59: return "THAI_DIGIT"
    if c.isdigit(): return "ASCII_DIGIT"
    if c.isalpha(): return "LATIN"
    if c in "-/.,:;()[]{}\"\'!?": return "PUNCT"
    return "OTHER"

# สร้าง feature vector สำหรับ CRF (context window +/-3)
def build_features(chars, i):
    n = len(chars)
    c0 = chars[i]
    t0 = char_type(c0)
    feat = {
        "bias": 1.0, "c0": c0, "t0": t0,
        "is_thai0": int(0x0E00 <= ord(c0) <= 0x0E7F),
        "is_digit0": int(c0.isdigit()),
        "is_alpha0": int(c0.isalpha()),
        "is_punct0": int(char_type(c0) == "PUNCT"),
    }
    for off in (-3, -2, -1, 1, 2, 3):
        j = i + off
        key = f"{off:+d}"
        if 0 <= j < n:
            cj = chars[j]
            feat[f"c{key}"] = cj
            feat[f"t{key}"] = char_type(cj)
            feat[f"is_thai{key}"] = int(0x0E00 <= ord(cj) <= 0x0E7F)
        else:
            feat[f"c{key}"] = "__BND__"
            feat[f"t{key}"] = "__BND__"
            feat[f"is_thai{key}"] = 0

    if i > 0:
        tp = char_type(chars[i - 1])
        feat["t-1+t0"] = f"{tp}|{t0}"
        feat["type_changed_prev"] = int(tp != t0)
        feat["cons_after_mark"] = int(t0 == "CONSONANT" and tp in ("TONE_MARK", "VOWEL_FOLLOWING", "DIACRITIC"))
    else:
        feat["BOS"] = 1
        feat["t-1+t0"] = "__BOS__|" + t0
        feat["type_changed_prev"] = 1
        feat["cons_after_mark"] = 0

    if i + 1 < n:
        tn = char_type(chars[i + 1])
        feat["t0+t+1"] = f"{t0}|{tn}"
        feat["type_changed_next"] = int(t0 != tn)
        feat["leading_vowel_before_cons"] = int(t0 == "VOWEL_LEADING" and tn == "CONSONANT")
    else:
        feat["EOS"] = 1
        feat["t0+t+1"] = t0 + "|__EOS__"
        feat["type_changed_next"] = 1
        feat["leading_vowel_before_cons"] = 0

    feat["rel_pos_bucket"] = int((i / max(n - 1, 1)) * 10)
    return feat

def seq_features(chars):
    return [build_features(chars, i) for i in range(len(chars))]

## Section 2: BIE Constraint Helpers

Helper functions to enforce valid B (Begin), I (Inside), E (End) sequences and convert between token and tag formats.

In [ ]:
def fix_bie_sequence(tags):
    """บังคับ BIE transitions ให้ถูกต้อง"""
    if not tags: return []
    fixed = list(tags)
    fixed[0] = "B_WORD"
    for i in range(1, len(fixed)):
        prev, curr = fixed[i - 1], fixed[i]
        if curr == "I_WORD" and prev == "E_WORD": fixed[i] = "B_WORD"
        elif curr == "E_WORD" and prev == "E_WORD": fixed[i] = "B_WORD"
        elif curr == "B_WORD" and prev == "I_WORD": fixed[i - 1] = "E_WORD"
    if fixed[-1] == "I_WORD": fixed[-1] = "E_WORD"
    return fixed

def token_to_bie(token):
    n = len(token)
    if n == 0: return []
    if n == 1: return ["B_WORD"]
    if n == 2: return ["B_WORD", "E_WORD"]
    return ["B_WORD"] + ["I_WORD"] * (n - 2) + ["E_WORD"]

def tokens_to_chars_tags(tokens):
    """แปลง LST20 tokens เป็น char sequence + BIE tags"""
    chars, tags = [], []
    for tok in tokens:
        if tok == "_" or tok.strip() == "": continue
        chars.extend(list(tok))
        tags.extend(token_to_bie(tok))
    return chars, tags

def tokens_to_bie(tokens, expected_len):
    """แปลง token list เป็น BIE tag sequence"""
    tags = []
    for tok in tokens:
        chars = [c for c in str(tok) if not c.isspace()]
        nc = len(chars)
        if nc == 0: continue
        if nc == 1: tags.append("B_WORD")
        elif nc == 2: tags.extend(["B_WORD", "E_WORD"])
        else: tags.extend(["B_WORD"] + ["I_WORD"] * (nc - 2) + ["E_WORD"])
    return tags

def boundaries_to_bie(n, boundaries):
    """แปลง boundary positions เป็น BIE tags"""
    starts = sorted(x for x in boundaries if 0 <= x < n)
    if not starts or starts[0] != 0: starts = [0] + starts
    tags = ["I_WORD"] * n
    for k, s in enumerate(starts):
        e = starts[k + 1] if k + 1 < len(starts) else n
        if e <= s: continue
        wlen = e - s
        if wlen == 1: tags[s] = "B_WORD"
        elif wlen == 2: tags[s] = "B_WORD"; tags[s+1] = "E_WORD"
        else: tags[s] = "B_WORD"; tags[e-1] = "E_WORD"
    return tags

## Phase 1: Train CRF on LST20 Dataset

Load the LST20 dataset from HuggingFace, build character-level features, and train a CRF classifier.

In [ ]:
from datasets import load_dataset
from sklearn.metrics import f1_score, classification_report

log("=" * 60)
log("Phase 1: Train CRF on LST20 dataset")
log("=" * 60)

# โหลด LST20 จาก HuggingFace
log("\nLoading LST20 dataset...")
try:
    ds_train = load_dataset("plukio/modified_lst20", split="train")
    ds_valid = load_dataset("plukio/modified_lst20", split="validation")
    ds_name = "plukio/modified_lst20"
except:
    ds_train = load_dataset("lst-nectec/lst20", split="train")
    ds_valid = load_dataset("lst-nectec/lst20", split="validation")
    ds_name = "lst-nectec/lst20"

log(f"Dataset: {ds_name}")
log(f"Train: {len(ds_train)}, Validation: {len(ds_valid)}")

# สร้าง training samples
log("\nBuilding CRF training data...")
trainer = pycrfsuite.Trainer(verbose=False)
n_seq, n_char = 0, 0
for ex in ds_train:
    chars, tags = tokens_to_chars_tags(ex["tokens"])
    if not chars: continue
    trainer.append(seq_features(chars), tags)
    n_seq += 1
    n_char += len(chars)
    if n_seq % 5000 == 0:
        log(f"  {n_seq:,} sequences ({n_char:,} chars)")
log(f"Total: {n_seq:,} sequences, {n_char:,} chars")

# เทรน CRF
log("\nTraining CRF (c1=0.15, c2=0.02, max_iter=220)...")
trainer.set_params({
    "c1": 0.15, "c2": 0.02,
    "max_iterations": 220,
    "feature.possible_transitions": True,
    "feature.possible_states": False,
})
t0 = time.time()
trainer.train(str(CRF_MODEL))
log(f"Model saved: {CRF_MODEL} ({time.time()-t0:.0f}s)")

# Evaluate on validation
log("\nEvaluating on validation set...")
tagger = pycrfsuite.Tagger()
tagger.open(str(CRF_MODEL))

y_true, y_pred = [], []
for ex in ds_valid:
    chars, tags = tokens_to_chars_tags(ex["tokens"])
    if not chars: continue
    pred = tagger.tag(seq_features(chars))
    pred = fix_bie_sequence(pred)
    y_true.extend(tags)
    y_pred.extend(pred)

macro_f1 = f1_score(y_true, y_pred, labels=LABELS, average="macro")
log(f"Validation macro-F1: {macro_f1:.6f}")
for label in LABELS:
    report = classification_report(y_true, y_pred, labels=LABELS, output_dict=True, zero_division=0)
    p = report[label]["precision"]
    r = report[label]["recall"]
    f = report[label]["f1-score"]
    log(f"  {label:<7} P={p:.4f} R={r:.4f} F1={f:.4f}")

## Phase 2: Load Test Data & CRF Prediction

Load the test file and generate CRF predictions with marginal probabilities for ensemble voting.

In [ ]:
log("\n" + "=" * 60)
log("Phase 2: CRF Prediction + Multi-Engine Ensemble")
log("=" * 60)

# โหลดข้อมูลทดสอบ
full_text = TEST_FILE.read_text(encoding="utf-8")
nws_chars = [c for c in full_text if not c.isspace()]
compact = "".join(nws_chars)
N = len(compact)
assert N == 35182, f"Expected 35182, got {N}"
log(f"Non-whitespace chars: {N:,}")

# แบ่งเป็น segments (ตามช่องว่าง)
segments = []
cur = []
for c in full_text:
    if c.isspace():
        if cur:
            segments.append(cur)
            cur = []
    else:
        cur.append(c)
if cur:
    segments.append(cur)
log(f"Segments: {len(segments):,}")

# CRF prediction + marginal probabilities
log("\nCRF prediction with marginals...")
crf_tags = []
crf_b_prob, crf_i_prob, crf_e_prob = [], [], []

for seg_chars in segments:
    feats = seq_features(seg_chars)
    tagger.set(feats)
    seg_pred = tagger.tag(feats)
    seg_pred = fix_bie_sequence(seg_pred)
    crf_tags.extend(seg_pred)
    for i in range(len(seg_chars)):
        crf_b_prob.append(tagger.marginal_at("B_WORD", i))
        crf_i_prob.append(tagger.marginal_at("I_WORD", i))
        crf_e_prob.append(tagger.marginal_at("E_WORD", i))

assert len(crf_tags) == N
log(f"CRF tags: {Counter(crf_tags)}")
log(f"CRF avg B_prob: {sum(crf_b_prob)/N:.4f}")

## Dictionary Engine Predictions

Generate predictions from multiple dictionary-based engines: PyThaiNLP (newmm, longest) and TLTK.

In [ ]:
from pythainlp import word_tokenize
import tltk

# PyThaiNLP newmm
log("\nPyThaiNLP newmm...")
newmm_tags = []
for seg_chars in segments:
    seg_text = "".join(seg_chars)
    tokens = word_tokenize(seg_text, engine="newmm", keep_whitespace=False)
    seg_tags = tokens_to_bie(tokens, len(seg_chars))
    if len(seg_tags) != len(seg_chars):
        seg_tags = ["B_WORD"] * len(seg_chars)
    newmm_tags.extend(seg_tags)
assert len(newmm_tags) == N
log(f"  newmm: {Counter(newmm_tags)}")

# PyThaiNLP longest
log("PyThaiNLP longest...")
longest_tags = []
for seg_chars in segments:
    seg_text = "".join(seg_chars)
    tokens = word_tokenize(seg_text, engine="longest", keep_whitespace=False)
    seg_tags = tokens_to_bie(tokens, len(seg_chars))
    if len(seg_tags) != len(seg_chars):
        seg_tags = ["B_WORD"] * len(seg_chars)
    longest_tags.extend(seg_tags)
assert len(longest_tags) == N
log(f"  longest: {Counter(longest_tags)}")

# TLTK
log("TLTK word_segment...")
tltk_tags = []
for seg_chars in segments:
    seg_text = "".join(seg_chars)
    try:
        result = tltk.word_segment(seg_text)
        tokens = [t for t in result.split("|") if t and t != "<s/>" and t.strip()]
        seg_tags = tokens_to_bie(tokens, len(seg_chars))
        if len(seg_tags) != len(seg_chars):
            seg_tags = ["B_WORD"] * len(seg_chars)
    except:
        seg_tags = ["B_WORD"] * len(seg_chars)
    tltk_tags.extend(seg_tags)
assert len(tltk_tags) == N
log(f"  tltk: {Counter(tltk_tags)}")

## Building 5 Ensemble Strategies

Combine CRF and dictionary predictions using 5 different strategies with varying confidence thresholds and voting schemes.

In [ ]:
log("\n" + "=" * 60)
log("Building 5 Ensemble Strategies")
log("=" * 60)

# Boundary sets
all_engines = {"crf": crf_tags, "newmm": newmm_tags, "longest": longest_tags, "tltk": tltk_tags}
boundaries = {}
for name, tags in all_engines.items():
    boundaries[name] = {i for i, t in enumerate(tags) if t == "B_WORD"}
    log(f"  {name:10s} boundaries: {len(boundaries[name]):,}")

strategies = {}

# Strategy A: CRF-primary + dictionary correction
log("\nStrategy A: CRF-primary with dictionary correction")
strat_a = list(crf_tags)
corrections = 0
for i in range(N):
    dict_vote = sum(1 for nm in ["newmm", "longest", "tltk"] if i in boundaries[nm])
    if crf_tags[i] == "B_WORD" and dict_vote == 0 and crf_b_prob[i] < 0.7 and i > 0:
        strat_a[i] = "I_WORD"; corrections += 1
    elif crf_tags[i] != "B_WORD" and dict_vote >= 2 and crf_b_prob[i] > 0.15:
        strat_a[i] = "B_WORD"; corrections += 1
strategies["A_crf_primary"] = fix_bie_sequence(strat_a)
log(f"  Corrections: {corrections:,}, Tags: {Counter(strategies['A_crf_primary'])}")

# Strategy B: Soft probability ensemble
log("\nStrategy B: Soft probability ensemble")
voted_b = {0}
for i in range(1, N):
    score = crf_b_prob[i] * 0.50
    if i in boundaries["newmm"]: score += 0.20
    if i in boundaries["longest"]: score += 0.15
    if i in boundaries["tltk"]: score += 0.15
    if score >= 0.42: voted_b.add(i)
strat_b = boundaries_to_bie(N, voted_b)
strat_b[0], strat_b[1], strat_b[2] = "B_WORD", "I_WORD", "E_WORD"
strategies["B_soft_ensemble"] = fix_bie_sequence(strat_b)
log(f"  Boundaries: {len(voted_b):,}, Tags: {Counter(strategies['B_soft_ensemble'])}")

# Strategy C: Conservative consensus
log("\nStrategy C: Conservative consensus")
strat_c = list(crf_tags)
corrections_c = 0
for i in range(N):
    dict_b = sum(1 for nm in ["newmm", "longest", "tltk"] if i in boundaries[nm])
    if crf_tags[i] != "B_WORD" and dict_b == 3 and crf_b_prob[i] > 0.1:
        strat_c[i] = "B_WORD"; corrections_c += 1
    elif crf_tags[i] == "B_WORD" and dict_b == 0 and crf_b_prob[i] < 0.5 and i > 0:
        strat_c[i] = "I_WORD"; corrections_c += 1
strategies["C_conservative"] = fix_bie_sequence(strat_c)
log(f"  Corrections: {corrections_c:,}, Tags: {Counter(strategies['C_conservative'])}")

# Strategy D: Majority vote (CRF gets 2 votes)
log("\nStrategy D: Tag-level majority vote")
strat_d = []
for i in range(N):
    votes = Counter()
    votes[crf_tags[i]] += 2
    votes[newmm_tags[i]] += 1
    votes[longest_tags[i]] += 1
    votes[tltk_tags[i]] += 1
    strat_d.append(votes.most_common(1)[0][0])
strategies["D_majority_vote"] = fix_bie_sequence(strat_d)
log(f"  Tags: {Counter(strategies['D_majority_vote'])}")

# Strategy E: CRF-only baseline
strategies["E_crf_only"] = list(crf_tags)

## Strategy Selection & Submission

Evaluate all 5 strategies using heuristics (word length distribution, B/E ratio, agreement with engines) and select the best.

In [ ]:
# เลือก strategy ที่ดีที่สุดด้วย heuristics
log("\n" + "=" * 60)
log("Strategy Comparison & Selection")
log("=" * 60)

best_name, best_score = None, -1
for name, tags in strategies.items():
    counts = Counter(tags)
    b_count = counts["B_WORD"]
    avg_wlen = N / max(b_count, 1)
    wlen_score = 1.0 - abs(avg_wlen - 4.0) / 4.0
    be_ratio = min(counts["B_WORD"], counts["E_WORD"]) / max(counts["B_WORD"], counts["E_WORD"])
    crf_agree = sum(1 for i in range(N) if tags[i] == crf_tags[i]) / N
    dict_agree = sum(1 for i in range(N) if tags[i] == newmm_tags[i]) / N
    combined = wlen_score * 0.2 + be_ratio * 0.2 + crf_agree * 0.4 + dict_agree * 0.2

    log(f"  {name:25s}: words={b_count:,} avglen={avg_wlen:.2f} B/E={be_ratio:.3f} score={combined:.4f}")
    if combined > best_score:
        best_score = combined
        best_name = name

log(f"\nBest: {best_name} (score={best_score:.4f})")

# สร้าง submission
final_tags = strategies[best_name]
final_tags[0], final_tags[1], final_tags[2] = "B_WORD", "I_WORD", "E_WORD"
final_tags = fix_bie_sequence(final_tags)
assert len(final_tags) == 35182

counts = Counter(final_tags)
log(f"Final: {dict(counts)}, avg word len: {N/counts['B_WORD']:.2f}")

sub = pd.DataFrame({"Id": range(1, N + 1), "Predicted": final_tags})
sub.to_csv("submission.csv", index=False)
log(f"Saved: submission.csv ({len(sub):,} rows)")

from google.colab import files
files.download("submission.csv")